# VLA 基础教程

欢迎来到 VLA (Vision-Language-Action) 学习之旅！

本教程将带你从零开始理解 VLA 模型。

## 1. 什么是 VLA？

**一句话解释**: VLA = 看懂图像 + 理解语言 + 执行动作

就像你看到桌上的苹果，听到'拿起苹果'的指令，然后伸手去拿。VLA 模型做的就是这件事。

In [ ]:
# 首先导入必要的库
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 环境准备完成！")

## 2. VLA 架构图解

```
图像输入 → [视觉编码器] → 视觉特征
                              ↓
文本指令 → [语言模型] → 语言特征 → [融合模块] → [动作头] → 动作输出
```

In [ ]:
# 可视化 VLA 架构
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

# 绘制模块
modules = [
    (1, 4, '图像输入\n(Image)'),
    (3, 4, '视觉编码器\n(Vision Encoder)'),
    (1, 2, '文本指令\n(Text)'),
    (3, 2, '语言模型\n(Language Model)'),
    (5, 3, '融合模块\n(Fusion)'),
    (7, 3, '动作头\n(Action Head)'),
    (9, 3, '动作输出\n(Action)'),
]

for x, y, label in modules:
    rect = plt.Rectangle((x-0.4, y-0.4), 0.8, 0.8, 
                         facecolor='lightblue', edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=9)

# 绘制箭头
arrows = [
    (1.4, 4, 2.6, 4),  # 图像 → 视觉编码器
    (1.4, 2, 2.6, 2),  # 文本 → 语言模型
    (3.4, 4, 4.6, 3.2),  # 视觉特征 → 融合
    (3.4, 2, 4.6, 2.8),  # 语言特征 → 融合
    (5.4, 3, 6.6, 3),  # 融合 → 动作头
    (7.4, 3, 8.6, 3),  # 动作头 → 输出
]

for x1, y1, x2, y2 in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

ax.set_title('VLA 架构流程图', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. 核心组件详解

### 3.1 视觉编码器

作用：把图像变成机器能理解的数字向量

In [ ]:
# 模拟视觉编码器
class SimpleVisionEncoder(nn.Module):
    def __init__(self, output_dim=768):
        super().__init__()
        # 简化的 ViT 结构
        self.patch_embed = nn.Conv2d(3, output_dim, kernel_size=16, stride=16)
        self.pos_embed = nn.Parameter(torch.randn(1, 196, output_dim))
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=output_dim, nhead=8),
            num_layers=2
        )
        self.output_dim = output_dim
    
    def forward(self, x):
        # x: (B, 3, 224, 224)
        x = self.patch_embed(x)  # (B, 768, 14, 14)
        x = x.flatten(2).transpose(1, 2)  # (B, 196, 768)
        x = x + self.pos_embed
        x = self.transformer(x)
        return x

# 测试视觉编码器
vision_encoder = SimpleVisionEncoder(output_dim=768)
dummy_image = torch.randn(2, 3, 224, 224)  # 2张224x224的图像
vision_features = vision_encoder(dummy_image)

print(f"输入图像形状: {dummy_image.shape}")
print(f"视觉特征形状: {vision_features.shape}")
print(f"特征维度: {vision_features.shape[-1]}")

### 3.2 语言模型

作用：理解文本指令的含义

In [ ]:
# 模拟语言模型
class SimpleLanguageModel(nn.Module):
    def __init__(self, vocab_size=1000, hidden_dim=768):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8),
            num_layers=2
        )
        self.output_dim = hidden_dim
    
    def forward(self, input_ids):
        # input_ids: (B, L)
        x = self.embedding(input_ids)  # (B, L, 768)
        x = self.transformer(x)
        return x

# 测试语言模型
language_model = SimpleLanguageModel(vocab_size=1000, hidden_dim=768)
dummy_tokens = torch.randint(0, 1000, (2, 20))  # 2个序列，每个20个token
language_features = language_model(dummy_tokens)

print(f"输入token形状: {dummy_tokens.shape}")
print(f"语言特征形状: {language_features.shape}")
print(f"特征维度: {language_features.shape[-1]}")

### 3.3 融合模块

作用：将视觉和语言特征融合

In [ ]:
# Cross-Attention 融合
class CrossAttentionFusion(nn.Module):
    def __init__(self, vision_dim=768, language_dim=768, hidden_dim=768):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        
        # 投影层
        self.vision_proj = nn.Linear(vision_dim, hidden_dim)
        self.language_proj = nn.Linear(language_dim, hidden_dim)
    
    def forward(self, vision_features, language_features):
        # vision_features: (B, N_v, D_v)
        # language_features: (B, N_l, D_l)
        
        # 投影到相同维度
        v = self.vision_proj(vision_features)
        l = self.language_proj(language_features)
        
        # Cross-attention: 用语言查询视觉
        fused, _ = self.cross_attn(query=l, key=v, value=v)
        fused = self.norm(fused)
        
        return fused

# 测试融合模块
fusion = CrossAttentionFusion(vision_dim=768, language_dim=768, hidden_dim=768)
fused_features = fusion(vision_features, language_features)

print(f"视觉特征: {vision_features.shape}")
print(f"语言特征: {language_features.shape}")
print(f"融合特征: {fused_features.shape}")

### 3.4 动作头

作用：根据融合特征生成具体动作

In [ ]:
# 简单的 MLP 动作头
class SimpleActionHead(nn.Module):
    def __init__(self, input_dim=768, action_dim=7, chunk_size=10):
        super().__init__()
        self.chunk_size = chunk_size
        self.action_dim = action_dim
        
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim * chunk_size)
        )
    
    def forward(self, x):
        # x: (B, N, D)
        # 取平均池化
        x = x.mean(dim=1)  # (B, D)
        actions = self.mlp(x)
        actions = actions.view(-1, self.chunk_size, self.action_dim)
        return actions

# 测试动作头
action_head = SimpleActionHead(input_dim=768, action_dim=7, chunk_size=10)
actions = action_head(fused_features)

print(f"融合特征: {fused_features.shape}")
print(f"预测动作: {actions.shape}")
print(f"动作含义: 预测未来 {actions.shape[1]} 步的动作，每步 {actions.shape[2]} 个关节")

## 4. 完整 VLA 模型

In [ ]:
# 组装完整 VLA 模型
class SimpleVLA(nn.Module):
    def __init__(self, action_dim=7, chunk_size=10):
        super().__init__()
        self.vision_encoder = SimpleVisionEncoder(output_dim=768)
        self.language_model = SimpleLanguageModel(vocab_size=1000, hidden_dim=768)
        self.fusion = CrossAttentionFusion(vision_dim=768, language_dim=768, hidden_dim=768)
        self.action_head = SimpleActionHead(input_dim=768, action_dim=action_dim, chunk_size=chunk_size)
    
    def forward(self, images, text_tokens):
        # 编码
        vision_features = self.vision_encoder(images)
        language_features = self.language_model(text_tokens)
        
        # 融合
        fused = self.fusion(vision_features, language_features)
        
        # 生成动作
        actions = self.action_head(fused)
        
        return actions

# 创建模型
vla_model = SimpleVLA(action_dim=7, chunk_size=10)

# 统计参数量
total_params = sum(p.numel() for p in vla_model.parameters())
print(f"✅ VLA 模型创建成功！")
print(f"总参数量: {total_params:,} ({total_params/1e6:.2f}M)")

## 5. 模拟推理

In [ ]:
# 模拟推理
with torch.no_grad():
    dummy_images = torch.randn(1, 3, 224, 224)
    dummy_text = torch.randint(0, 1000, (1, 20))
    
    predicted_actions = vla_model(dummy_images, dummy_text)
    
    print(f"输入: 1张图像 + '拿起红色方块'")
    print(f"输出动作形状: {predicted_actions.shape}")
    print(f"动作范围: [{predicted_actions.min():.2f}, {predicted_actions.max():.2f}]")
    
    # 可视化预测的动作
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # 动作序列热力图
    im = axes[0].imshow(predicted_actions[0].numpy(), cmap='RdBu_r', aspect='auto')
    axes[0].set_xlabel('关节维度')
    axes[0].set_ylabel('时间步')
    axes[0].set_title('预测动作序列')
    plt.colorbar(im, ax=axes[0])
    
    # 每个关节的动作曲线
    for i in range(min(7, predicted_actions.shape[2])):
        axes[1].plot(predicted_actions[0, :, i].numpy(), label=f'关节 {i}')
    axes[1].set_xlabel('时间步')
    axes[1].set_ylabel('动作值')
    axes[1].set_title('各关节动作曲线')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. 总结

本教程展示了 VLA 的核心组件：

1. **视觉编码器**: 将图像转换为特征向量
2. **语言模型**: 理解文本指令
3. **融合模块**: 结合视觉和语言信息
4. **动作头**: 生成机器人动作

### 下一步

- 学习 `02_training.ipynb` 了解如何训练 VLA
- 学习 `03_inference.ipynb` 了解如何部署推理
- 阅读 `TUTORIAL.md` 获取更详细的理论解释

In [ ]:
print("🎉 恭喜完成 VLA 基础教程！")
print("\n关键概念:")
print("- VLA = Vision + Language + Action")
print("- 动作块: 一次性预测多步动作")
print("- 融合方式: Cross-Attention 效果最好")
print("\n继续学习: notebooks/02_training.ipynb")